In [1]:
# split datasets
import random

# 读取数据
words = open('names.txt', 'r').read().splitlines()

# 固定随机种子，保证每次划分结果一致
random.seed(42)
random.shuffle(words)

# 按照 80%、10%、10% 划分
n1 = int(0.8 * len(words))
n2 = int(0.9 * len(words))

words_train = words[:n1]
words_val = words[n1:n2]
words_test = words[n2:]

print(len(words_train), len(words_val), len(words_test))
words_train[:10], words_test[:5], words_val[:5]

25626 3203 3204


(['yuheng',
  'diondre',
  'xavien',
  'jori',
  'juanluis',
  'erandi',
  'phia',
  'samatha',
  'phoenix',
  'emmelynn'],
 ['mustafa', 'reuben', 'kahlel', 'balian', 'izael'],
 ['amay', 'aytana', 'jenevi', 'zylen', 'naim'])

In [2]:
import torch
import torch.nn.functional as F

N = torch.zeros((27*27, 27), dtype=torch.int32)  # 27*27 situation for taking two characters 

chars = ['.'] + sorted(set(''.join(words)))
stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for ch, i in stoi.items()}

In [14]:
def build_trigram_dataset(words):
    xs, ys = [], []

    for w in words:
        chs = ['.', '.'] + list(w) + ['.']

        for ch1, ch2, ch3 in zip(chs, chs[1:], chs[2:]):
            ix1 = stoi[ch1] * 27 + stoi[ch2]
            ix2 = stoi[ch3]
            xs.append(ix1)
            ys.append(ix2)

    return torch.tensor(xs), torch.tensor(ys)


xs, ys = build_trigram_dataset(words_train)
xval3, yval3 = build_trigram_dataset(words_val)
xtest3, ytest3 = build_trigram_dataset(words_test)

num = xs.nelement()
print('number of training examples:', num)

g = torch.Generator().manual_seed(2147483647)
W = torch.randn((27 * 27, 27), generator=g, requires_grad=True)

number of training examples: 182625


In [18]:
def evaluate_trigram(xs, ys, W):
    logits = W[xs]
    counts = logits.exp()
    probs = counts / counts.sum(1, keepdim=True)

    loss = -probs[torch.arange(len(ys)), ys].log().mean()
    return loss.item()

In [19]:
strengths = [
    0.0,
    0.0001,
    0.001,
    0.01,
    0.1,
    1.0
]

results = []
best_val_loss = float('inf')
best_strength = None
best_W = None

for smth in strengths:
    # 每次使用相同的初始化
    g = torch.Generator().manual_seed(2147483647)
    W = torch.randn(
        (27 * 27, 27),
        generator=g,
        requires_grad=True
    )

    for i in range(100):
        logits = W[xs]
        counts = logits.exp()
        probs = counts / counts.sum(1, keepdim=True)

        nll = -probs[torch.arange(num), ys].log().mean()
        loss = nll + smth * (W**2).mean()

        W.grad = None
        loss.backward()
        W.data += -100 * W.grad

    train_loss = evaluate_trigram(xs, ys, W)
    val_loss = evaluate_trigram(xval3, yval3, W)

    results.append((smth, train_loss, val_loss))

    print(
        f'smth={smth:<6} '
        f'train={train_loss:.4f} '
        f'val={val_loss:.4f}'
    )

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_strength = smth
        best_W = W.detach().clone()

smth=0.0    train=2.3799 val=2.3938
smth=0.0001 train=2.3799 val=2.3938
smth=0.001  train=2.3799 val=2.3937
smth=0.01   train=2.3798 val=2.3936
smth=0.1    train=2.3797 val=2.3926
smth=1.0    train=2.4243 val=2.4316


In [21]:
test_loss = evaluate_trigram(xtest3, ytest3, best_W)

print('Best smoothing:', best_strength)
print('Best val loss:', best_val_loss)
print('Final test loss:', test_loss)

Best smoothing: 0.1
Best val loss: 2.3926165103912354
Final test loss: 2.389838933944702
